In [1]:
import os
import gc
import re
import time
import sqlite3
import requests 
import numpy as np
import pandas as pd
import urllib.request
from bs4 import BeautifulSoup

In [2]:
def get_soup(url):
    '''
    Create a soup object of a web url

    Args: 
        url - String (ex. https://basketball-reference.com/)

    Returns:
        soup - BeautifulSoup object
    '''
    try:
        with urllib.request.urlopen(url) as response:
            html = response.read()
    except urllib.error.URLError as e:
        print(f"Error fetching URL: {e.reason}")

    ### create the soup object
    soup = BeautifulSoup(html, 'html.parser')

    return soup

https://www.basketball-reference.com/

Teams:

Eastern Conference:
- ATL — Atlanta Hawks
- BOS — Boston Celtics
- BRK — Brooklyn Nets
- CHI — Chicago Bulls
- CHO — Charlotte Hornets
- CLE — Cleveland Cavaliers
- DET — Detroit Pistons
- IND — Indiana Pacers
- MIA — Miami Heat
- MIL — Milwaukee Bucks
- NYK — New York Knicks
- ORL — Orlando Magic
- PHI — Philadelphia 76ers
- TOR — Toronto Raptors
- WAS — Washington Wizards

Western Conference:
- DAL — Dallas Mavericks
- DEN — Denver Nuggets
- GSW — Golden State Warriors
- HOU — Houston Rockets
- LAC — Los Angeles Clippers
- LAL — Los Angeles Lakers
- MEM — Memphis Grizzlies
- MIN — Minnesota Timberwolves
- NOP — New Orleans Pelicans
- OKC — Oklahoma City Thunder
- PHO — Phoenix Suns
- POR — Portland Trail Blazers
- SAC — Sacramento Kings
- SAS — San Antonio Spurs
- UTA — Utah Jazz


Players:
- Basketball-Reference encodes player names as: /(last initial)/(first 5 letters of last name)(first 2 letters of first name)01.html
  - i.e. `j/jamesle01.html`
  - Names with the same last initial and first name will increment that last digits of the encoded name as they entered into the BR database.

### Player ID Scraping
- Create a function which pulls the player ID Basketball-Reference associates to each player from their team data.

In [3]:
def get_player_ids(team, year):
    url = f'https://www.basketball-reference.com/teams/{team}/{year}.html'

    soup = get_soup(url)

    ids = []
    players = {}

    rows = soup.find("tbody").find_all("tr")

    for row in rows:
        player_cell = row.find("td", {"data-stat": "player"})
        
        if player_cell:
            link = player_cell.find("a")
            
            if link:
                name = link.text
                href = link.get("href")
                split = href.split('/')[-1]
                if '.html' in split:
                    split = split[0:split.find('.')]
                    players[name] = split
                else:
                    players[name] = href
                    print('Check Player List')

    if not players:
        print('No Team Found')
    else:
        # print(ids)
        return players
    
kings = get_player_ids('SAC', 2026)
kings

{'DeMar DeRozan': 'derozde01',
 'Nique Clifford': 'cliffni01',
 'Maxime Raynaud': 'raynama01',
 'Precious Achiuwa': 'achiupr01',
 'Russell Westbrook': 'westbru01',
 'Malik Monk': 'monkma01',
 'Dylan Cardwell': 'cardwdy01',
 'Drew Eubanks': 'eubandr01',
 'Zach LaVine': 'lavinza01',
 'Devin Carter': 'cartede02',
 'Daeqwon Plowden': 'plowdda01',
 'Doug McDermott': 'mcderdo01',
 'Keegan Murray': 'murrake02',
 'Killian Hayes': 'hayeski01',
 'Domantas Sabonis': 'sabondo01',
 'Patrick Baldwin Jr.': 'baldwpa01',
 'Isaiah Stevens': 'steveis01',
 "De'Andre Hunter": 'huntede01'}

In [ ]:
def get_player_data(username):
    url = f"https://www.basketball-reference.com/players/{username[0]}/{username}.html"

    soup = get_soup(url)
    tbodies = soup.find_all("tbody")

    tables = {}

    for i, tbody in enumerate(tbodies):
        rows_data = []
        table_name = None

        for row in tbody.find_all("tr"):
            row_id = row.get("id")

            if row_id and table_name is None:
                table_name = row_id.split(".")[0]

            cells = row.find_all(["th", "td"])

            row_dict = {}
            for j, cell in enumerate(cells):
                col_name = cell.get("data-stat")
                text = cell.get_text(strip=True)

                if col_name is None:
                    col_name = f"col_{j}"

                row_dict[col_name] = text

            if row_dict:
                rows_data.append(row_dict)

        df = pd.DataFrame(rows_data)
        df["player"] = username

        if table_name is None:
            table_name = f"table_{i}"

            if table_name == "table_0" and df.shape[0] == 5:
                table_name = "past_5_games"
            elif table_name == "table_3" and df.shape[0] == 7:
                table_name = "stathead_insight"

        tables[table_name] = df

    ## For Testing May Remove
    for name, df in tables.items():
        print(f"{name}: {df.shape}")

    ### Insert to Database+

    with sqlite3.connect("example.db", timeout=60) as conn:
        for key, df in tables.items():
            table_name = f"{username}_{key}"

            df.to_sql(
                table_name,
                conn,
                if_exists="replace",
                index=False
            )

    return tables

data = get_player_data("westbru01")

past_5_games: (5, 28)
per_game_stats: (20, 32)
per_game_stats_post: (16, 32)
stathead_insight: (7, 4)
totals_stats: (20, 33)
totals_stats_post: (16, 33)
per_minute_stats: (20, 32)
per_minute_stats_post: (16, 32)
advanced: (20, 30)
advanced_post: (16, 30)
shooting: (20, 32)
shooting_post: (16, 32)


In [8]:
with sqlite3.connect('example.db') as conn:
    df = pd.read_sql_query("SELECT * FROM westbru01_per_game_stats", conn)

df

,year_id,age,team_name_abbr,comp_name_abbr,pos,games,games_started,mp_per_g,fg_per_g,fga_per_g,...,drb_per_g,trb_per_g,ast_per_g,stl_per_g,blk_per_g,tov_per_g,pf_per_g,pts_per_g,awards,player
0,2008-09,20,OKC,NBA,PG,82,65,32.5,5.3,13.4,...,2.7,4.9,5.3,1.3,0.2,3.3,2.3,15.3,ROY-4,westbru01
1,2009-10,21,OKC,NBA,PG,82,82,34.3,5.9,14.1,...,3.1,4.9,8.0,1.3,0.4,3.3,2.5,16.1,,westbru01
2,2010-11,22,OKC,NBA,PG,82,82,34.7,7.5,17.0,...,3.1,4.6,8.2,1.9,0.4,3.9,2.5,21.9,"AS,NBA2",westbru01
3,2011-12,23,OKC,NBA,PG,66,66,35.3,8.8,19.2,...,3.1,4.6,5.5,1.7,0.3,3.6,2.2,23.6,"MVP-12,DPOY-18,AS,NBA2",westbru01
4,2012-13,24,OKC,NBA,PG,82,82,34.9,8.2,18.7,...,3.9,5.2,7.4,1.8,0.3,3.3,2.3,23.2,"MVP-9,DPOY-17,AS,NBA2",westbru01
5,2013-14,25,OKC,NBA,PG,46,46,30.7,7.5,17.2,...,4.5,5.7,6.9,1.9,0.2,3.8,2.3,21.8,,westbru01
6,2014-15,26,OKC,NBA,PG,67,67,34.4,9.4,22.0,...,5.4,7.3,8.6,2.1,0.2,4.4,2.7,28.1,"MVP-4,AS,NBA2",westbru01
7,2015-16,27,OKC,NBA,PG,80,80,34.4,8.2,18.1,...,6.0,7.8,10.4,2.0,0.3,4.3,2.5,23.5,"MVP-4,AS,NBA1",westbru01
8,2016-17,28,OKC,NBA,PG,81,81,34.6,10.2,24.0,...,9.0,10.7,10.4,1.6,0.4,5.4,2.3,31.6,"MVP-1,AS,NBA1",westbru01
9,2017-18,29,OKC,NBA,PG,80,80,36.4,9.5,21.1,...,8.2,10.1,10.3,1.8,0.3,4.8,2.5,25.4,"MVP-5,AS,NBA2",westbru01
